# Inference Test Data 
This notebook is created for testing dataset in batch inference with WER metrics.
Before running this notebook, ensure you have already done following setup:
- install dependencies from `requirements.txt`
- store your video or audio files in folder data, then separated each type of files, ex: video (.mp4), audio (.wav), if you already preprocess mouth_roi (.mp4) and want to use it for inference, store in folder data/mouth_roi. Note that if we use OG demo.py for inference, mouth cropped video does not include audio channel. Use scripts.extract_audio and scripts.merge_audio_video for your convenience 
- in csv folder write proper address path of your video or audio and corresponding text for WER evaluation
- Don't forget to update and check conf/config.yaml to ensure your inference properly set up. (You also can set in notebook below manually, default "av")
- set .env parameter for wandb logging
- This notebook use fine-tuned model: huge_high_resource_lrs2lrs3vox2avsp.pth
and resnet backbone: resnet_transformer_huge (step 7 cell) 

## 1) Set Up Environment and Dependencies

In [ ]:
from pathlib import Path
import json
import os
from metrics import WER
from typing import Dict, Any, Optional, Tuple
import datetime
import importlib
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from omegaconf import OmegaConf
from hydra import initialize_config_dir, compose
import utils.inference_ as inference_module
from metrics import get_wer

inference_module = importlib.reload(inference_module)
transcribe = inference_module.transcribe

try:
    import wandb
    HAS_WANDB = True
except Exception:
    HAS_WANDB = False

print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
print('wandb available:', HAS_WANDB)
print('inference_.py reloaded for latest transcribe() changes')

torch: 2.5.1+cu121
cuda available: True
wandb available: True
inference_.py reloaded for latest transcribe() changes


In [ ]:
import wandb
wandb.login()

## 2) Define Configuration and Input Contracts

We define file paths, runtime options, and expected CSV schema.

In [ ]:
ROOT = Path(".").resolve()
CSV_IN = ROOT / "path/to/your/input.csv"
CSV_OUT = ROOT / "path/to/your/output.csv"
CHECKPOINT_PATH = ROOT / "path/to/checkpoint.json"
SAVE_EVERY = 20
MAX_ROWS = -1 # set to 0 or negative number to process all rows, otherwise for dry-run try small number
timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
modality = "av" # change to "av" for audio-visual, "a" for audio-only, "v" for visual-only
with initialize_config_dir(config_dir=str((ROOT / "conf").resolve()), version_base=None):
    cfg = compose(config_name="config")

detector = str(cfg.get("detector", "mediapipe"))
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

assert "model" in cfg and "pretrained_model_path" in cfg.model, "Missing cfg.model.pretrained_model_path"
print("Model ckpt:", cfg.model.pretrained_model_path)

EXPECTED_COLUMNS = ['dataset', 'video_path', 'audio_path','true_text', 'status', 'prediction_text', 'wer']

print('CSV_IN:', CSV_IN)
print('CSV_OUT:', CSV_OUT)
print('CHECKPOINT_PATH:', CHECKPOINT_PATH)
print('modality:', modality, 'detector:', detector, 'device:', device)

## 3) Create Core Data Structures and Helper Utilities

In [ ]:
def normalize_text(text: str) -> str:
    return ' '.join(str(text).strip().lower().split())

def load_checkpoint(path: Path) -> Dict[str, Any]:
    if path.exists():
        return json.loads(path.read_text(encoding='utf-8'))
    return {'processed': {}, 'last_idx': -1}

def save_checkpoint(path: Path, payload: Dict[str, Any]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding='utf-8')

def resolve_video_path(video_path_value: str) -> Path:
    p = Path(str(video_path_value).strip())
    if p.is_absolute():
        return p
    return (ROOT / p).resolve()

def resolve_audio_path(audio_path_value: str) -> Path:
    p = Path(str(audio_path_value).strip())
    if p.is_absolute():
        return p
    return (ROOT / p).resolve()

def resolve_media_paths(row: pd.Series, current_modality: str) -> Tuple[Optional[Path], Optional[Path], str]:
    rel_video = str(row.get('video_path', '')).strip()
    rel_audio = str(row.get('audio_path', '')).strip()
    video_path = resolve_video_path(rel_video) if rel_video else None
    audio_path = resolve_audio_path(rel_audio) if rel_audio else None

    if current_modality == 'a':
        key = rel_audio
    else:
        key = rel_video
    return video_path, audio_path, key

# quick sanity checks
assert normalize_text('  A   B  ') == 'a b'
print('Helper checks passed.')

Helper checks passed.


## 4) Implement the Main Processing Function

Load CSV, restore checkpoint/previous output, and run resumable batched inference.

In [ ]:
def read_test_csv(path: Path) -> pd.DataFrame:
    df_local = pd.read_csv(path, dtype='object', keep_default_na=False)
    missing = [col for col in EXPECTED_COLUMNS if col not in df_local.columns]
    if missing:
        raise ValueError(f'Missing required columns in {path}: {missing}')
    return df_local

df = read_test_csv(CSV_IN)
checkpoint_payload = load_checkpoint(CHECKPOINT_PATH)
processed_meta = dict(checkpoint_payload.get('processed', {}))
print('Loaded processed_meta:', len(processed_meta))

if MAX_ROWS > 0:
    df = df.head(MAX_ROWS).copy()

for c in ['prediction_text', 'status']:
    if c not in df.columns:
        df[c] = ''
    df[c] = df[c].astype('object')

if 'wer' not in df.columns:
    df['wer'] = float('nan')
df['wer'] = pd.to_numeric(df['wer'], errors='coerce')

Loaded processed_meta: 0


## 5) Add Validation and Error Handling

In [8]:
assert CSV_IN.exists(), f'Missing input CSV: {CSV_IN}'
assert 'true_text' in df.columns
target_col = 'audio_path' if modality == 'a' else 'video_path'
assert target_col in df.columns, f'Missing required column: {target_col}'

missing_paths = 0
for value in df[target_col].astype(str).head(50):
    if modality == 'a':
        exists = resolve_audio_path(value).exists()
    else:
        exists = resolve_video_path(value).exists()
    if not exists:
        missing_paths += 1

print(f'Sample missing {target_col} files (first 50 rows):', missing_paths)
print('Validation checks complete.')

Sample missing audio_path files (first 50 rows): 0
Validation checks complete.


## 6) Write Quick Unit Tests for Core Behavior

In [ ]:
# Quick tests normalizing to lowercase and collapsing spaces
assert normalize_text('HELLO   WORLD') == 'hello world'
assert isinstance(load_checkpoint(Path('does_not_exist.json')), dict)
print('Quick tests passed.')

Quick tests passed.


## 7) Run an End-to-End Example in Notebook Cells

This cell supports resume/checkpointing + optional W&B logging.

In [ ]:
# Load pretrained model and backbone

with initialize_config_dir(config_dir=str((ROOT / "conf").resolve()), version_base=None):
    cfg = compose(
        config_name="config",
        overrides=[
            "model/backbone=resnet_transformer_huge",
            "model.pretrained_model_path=models/huge_high_resource_lrs2lrs3vox2avsp.pth",
        ],
    )
print("backbone:", cfg.model.backbone)
print("pretrained_model_path:", cfg.model.pretrained_model_path)

backbone: {'_target_': 'espnet.nets.pytorch_backend.e2e_asr_transformer.E2E', 'odim': 1049, 'idim': 512, 'adim': 1280, 'aheads': 16, 'eunits': 5120, 'elayers': 32, 'ddim': '${model.backbone.adim}', 'dheads': '${model.backbone.aheads}', 'dunits': '${model.backbone.eunits}', 'dlayers': 9, 'gamma_init': 0.1}
pretrained_model_path: models/huge_high_resource_lrs2lrs3vox2avsp.pth


In [ ]:
run = None
if HAS_WANDB:
    wandb.login(relogin=False)
    run = wandb.init(
        project='usr2-inference',
        name=f'{modality}_run_{timestamp}',
        settings=wandb.Settings(save_code=False),
        reinit=True,
    )

save_counter = 0

try:
    for idx, row in tqdm(df.iterrows(), total=len(df), desc='Batched inference'):
        if str(df.at[idx, 'status']).strip().lower() == 'ok':
            continue

        video_path, audio_path, row_key = resolve_media_paths(row, modality)
        true_text = normalize_text(row.get('true_text', ''))

        if modality == 'a':
            missing = audio_path is None or not audio_path.exists()
            missing_msg = 'audio file not found'
        elif modality == 'v':
            missing = video_path is None or not video_path.exists()
            missing_msg = 'video file not found'
        else:
            missing = (video_path is None or not video_path.exists()) or (audio_path is None or not audio_path.exists())
            missing_msg = 'video/audio file not found'

        if missing:
            df.at[idx, 'prediction_text'] = missing_msg
            df.at[idx, 'wer'] = np.nan
            df.at[idx, 'status'] = 'error'
            processed_meta[row_key] = {'idx': int(idx), 'status': 'error', 'error': missing_msg}
            continue

        try:
            pred_text = transcribe(
                video_path=str(video_path) if video_path is not None else None,
                audio_path=str(audio_path) if audio_path is not None else None,
                cfg=cfg,
                modality=modality,
                device=device,
                detector=detector,
            )
            pred_norm = normalize_text(pred_text)
            row_wer = float(get_wer(pred_norm, true_text)) if true_text else np.nan

            df.at[idx, 'prediction_text'] = pred_norm
            df.at[idx, 'wer'] = row_wer
            df.at[idx, 'status'] = 'ok'

            processed_meta[row_key] = {'idx': int(idx), 'status': 'ok'}

            if run is not None and not np.isnan(row_wer):
                wandb.log({'wer_row': row_wer, 'processed_rows': int((df['status'] == 'ok').sum())})

        except Exception as exc:
            df.at[idx, 'prediction_text'] = f'<error: {type(exc).__name__}: {exc}>'
            df.at[idx, 'wer'] = np.nan
            df.at[idx, 'status'] = 'error'
            processed_meta[row_key] = {'idx': int(idx), 'status': 'error', 'error': str(exc)}

        save_counter += 1
        if save_counter >= SAVE_EVERY:
            CSV_OUT.parent.mkdir(parents=True, exist_ok=True)
            df.to_csv(CSV_OUT, index=False)
            save_checkpoint(CHECKPOINT_PATH, {'processed': processed_meta, 'last_idx': int(idx)})
            save_counter = 0

except KeyboardInterrupt:
    print('Interrupted by user. Saving resume state...')

finally:
    CSV_OUT.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(CSV_OUT, index=False)
    save_checkpoint(CHECKPOINT_PATH, {'processed': processed_meta})

    from metrics import WER
    wer_metric = WER()

    status_series = df['status'].astype(str).str.strip().str.lower()
    valid_mask = status_series != 'error'
    eval_df = df.loc[valid_mask].copy()

    used_rows = 0
    for _, eval_row in eval_df.iterrows():
        pred_text = normalize_text(eval_row.get('prediction_text', ''))
        true_text = normalize_text(eval_row.get('true_text', ''))
        if not pred_text or not true_text:
            continue
        wer_metric.update(pred_text, true_text)
        used_rows += 1

    aggregate_wer = float(wer_metric.compute().item()) if used_rows > 0 else np.nan

    print('Saved output:', CSV_OUT)
    print('Saved checkpoint:', CHECKPOINT_PATH)
    print('Completed rows:', int((df['status'] == 'ok').sum()), '/', len(df))
    print('Rows used for aggregate WER:', used_rows)
    print('Aggregate WER:', aggregate_wer)

    if run is not None:
        wandb.log({'wer_aggregate': aggregate_wer, 'completed_rows': int((df['status'] == 'ok').sum())})
        run.finish()

df.head()

In [ ]:
# quick sanity check on WER calculation
import jiwer

refs = ["hello world my name is John", "the quick brown fox", "lips are moving", "quick brown fox jumps over the stone"]
hyps = ["hello world my name is John", "the quick brown", "lips are moving very fast", "quick"]

row_wer = [jiwer.wer(r, h) for r, h in zip(refs, hyps)]
number_of_rows = len(refs)
aggregate_wer = float(jiwer.wer(refs, hyps))
print('Row-wise WER:', row_wer)
print('Average WER:', sum(row_wer) / len(row_wer))
print('Aggregate WER:', aggregate_wer)

Row-wise WER: [0.16666666666666666, 0.25, 0.6666666666666666, 0.8571428571428571]
Average WER: 0.4851190476190476
Aggregate WER: 0.5


In [ ]:
# Implementation using metrics library metrics WER Calculations
wer_metric = WER()

status_series = df["status"].astype(str).str.strip().str.lower()
valid_mask = status_series != "error"   # ignore status error rows only status ok counts
eval_df = df.loc[valid_mask].copy()

used_rows = 0
for _, row in eval_df.iterrows():
    pred_text = normalize_text(row.get("prediction_text", ""))
    true_text = normalize_text(row.get("true_text", ""))
    if not pred_text or not true_text:
        continue
    wer_metric.update(pred_text, true_text)
    used_rows += 1

aggregate_wer = 100 * float(wer_metric.compute().item()) if used_rows > 0 else float("nan")

print(f"Rows total: {len(df)}")
print(f"Rows ignored (status=error): {(~valid_mask).sum()}")
print(f"Rows used for corpus WER: {used_rows}")
print(f"Aggregate WER: {aggregate_wer:.4f}%")

Rows total: 1243
Rows ignored (status=error): 0
Rows used for corpus WER: 1243
Aggregate WER: 1.7868%


In [ ]:
# Inspect some error cases
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from hydra import initialize_config_dir, compose
from utils.inference_ import transcribe
from metrics import get_wer

ROOT = Path('path/to/home/')
CSV_PATH = ROOT / f'csv/{modality}_run_{timestamp}.csv'

def normalize_text(text: str) -> str:
    return ' '.join(str(text).strip().lower().split())

df = pd.read_csv(CSV_PATH, dtype='object', keep_default_na=False)
df['wer'] = pd.to_numeric(df.get('wer', np.nan), errors='coerce')

with initialize_config_dir(config_dir=str((ROOT / 'conf').resolve()), version_base=None):
    cfg = compose(
        config_name='config',
        overrides=[
            'model/backbone=resnet_transformer_huge',
            'model.pretrained_model_path=models/huge_high_resource_lrs2lrs3vox2avsp.pth',
        ],
    )

detector = str(cfg.get('detector', 'mediapipe'))
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
cp_path = ROOT / f'csv/{modality}_checkpoint_{timestamp}.json' # path to your checkpoint file

idxs = ()
if cp_path.exists():
    try:
        cp = json.loads(cp_path.read_text(encoding='utf-8'))
        processed = cp.get('processed', {})
        idxs = sorted({int(v['idx']) for v in processed.values() if isinstance(v, dict) and v.get('status') == 'error' and 'idx' in v})
    except Exception as exc:
        print('Failed to read checkpoint:', exc)
        idxs = ()
else:
    idxs = ()

for idx in idxs:
    if idx not in df.index:
        print(idx, 'missing-index')
        continue
    if str(df.at[idx, 'status']).strip().lower() != 'error':
        print(idx, f"skip-status={df.at[idx, 'status']}")
        continue

    rel_video = str(df.at[idx, 'video_path']).strip() if 'video_path' in df.columns else ''
    rel_audio = str(df.at[idx, 'audio_path']).strip() if 'audio_path' in df.columns else ''
    video_path = (ROOT / rel_video).resolve() if rel_video and not Path(rel_video).is_absolute() else (Path(rel_video) if rel_video else None)
    audio_path = (ROOT / rel_audio).resolve() if rel_audio and not Path(rel_audio).is_absolute() else (Path(rel_audio) if rel_audio else None)
    true_text = normalize_text(df.at[idx, 'true_text'])

    try:
        pred = transcribe(
            video_path=str(video_path) if video_path is not None else None,
            audio_path=str(audio_path) if audio_path is not None else None,
            cfg=cfg,
            modality=modality,
            device=device,
            detector=detector,
        )
        pred_norm = normalize_text(pred)
        row_wer = float(get_wer(pred_norm, true_text)) if true_text else np.nan

        df.at[idx, 'prediction_words'] = pred_norm
        df.at[idx, 'wer'] = row_wer
        df.at[idx, 'status'] = 'ok'
        df.at[idx, 'error'] = ''
        print(idx, 'ok', row_wer)
    except Exception as exc:
        df.at[idx, 'prediction_words'] = f'<error: {type(exc).__name__}: {exc}>'
        df.at[idx, 'wer'] = np.nan
        df.at[idx, 'status'] = 'error'
        df.at[idx, 'error'] = str(exc)
        print(idx, 'error', exc)

df.to_csv(CSV_PATH, index=False)
print('saved:', CSV_PATH)